# 08 Matching and Scoring

## Introduction

This notebook compares the structured CV data with the structured job posting data.

The goal of this step is to calculate how well a candidate profile matches a specific IT job posting. The matching is based only on previously extracted structured information from the CV and the job advertisement.

The notebook calculates separate scores for:
- required skills
- nice-to-have skills
- technologies
- experience requirements
- education requirements
- certifications
- language requirements
- soft skills

This notebook uses a hybrid matching approach.

Rule-based matching is used for objective and clearly measurable requirements such as technical skills, tools, programming languages, experience years, education, certifications and language requirements.

LLM-based semantic analysis is used for less direct requirements such as responsibilities alignment, soft skills evidence and overall role fit.

The final score combines both approaches in order to keep the result explainable, while still allowing semantic interpretation where exact keyword matching is not sufficient.

## 1. Imports and Settings

In [47]:
import os
import json
import re
import pandas as pd

from pathlib import Path
from datetime import datetime
from typing import List

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [48]:
CV_EXTRACTION_DIR = Path("../outputs/cv_extraction")
JOB_EXTRACTION_DIR = Path("../outputs/job_extraction")
MATCHING_DIR = Path("../outputs/matching")
load_dotenv(dotenv_path=Path("../.env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


structured_cv_path = CV_EXTRACTION_DIR / "structured_cv.json"
structured_job_path = JOB_EXTRACTION_DIR / "structured_job.json"

matching_result_path = MATCHING_DIR / "matching_result.json"
matching_report_path = MATCHING_DIR / "matching_report.md"

## 2. LLM Settings

In [49]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

## 3. Load Structured CV and Job Posting

In [50]:
with open(structured_cv_path, "r", encoding="utf-8") as file:
    structured_cv = json.load(file)

In [51]:
with open(structured_job_path, "r", encoding="utf-8") as file:
    structured_job = json.load(file)

## 4. Define Text Normalization Functions

In [52]:
def normalize_text(text):
    if text is None:
        return ""

    text = str(text).lower().strip()

    replacements = {
        "asp.net": "aspnet",
        "angular.js": "angular",
        "node.js": "nodejs",
        "vue.js": "vue",
        "react.js": "react",
        "c#": "csharp",
        "c++": "cplusplus",
        ".net": "dotnet",
        "html 5": "html5",
        "j son": "json",
        "json": "json",
        "js": "javascript",
        "ts": "typescript",
        "postgres": "postgresql",
        "postgre sql": "postgresql",
        "gcp": "google cloud platform"
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [53]:
def contains_normalized_phrase(text, phrase):
    if not text or not phrase:
        return False

    phrase_tokens = phrase.split()

    if len(phrase_tokens) == 1:
        pattern = r"\b" + re.escape(phrase) + r"\b"
    else:
        pattern = r"\b" + r"\s+".join([re.escape(token) for token in phrase_tokens]) + r"\b"

    return re.search(pattern, text) is not None

## 5. Define Helper Functions for Lists and Dictionaries

In [54]:
def ensure_list(value):
    if value is None:
        return []

    if isinstance(value, list):
        return value

    return [value]

In [55]:
def extract_text_from_item(item):
    if item is None:
        return ""

    if isinstance(item, str):
        return item

    if isinstance(item, dict):
        values = []

        for key, value in item.items():
            if value is None:
                continue

            if key in ["is_required", "metadata"]:
                continue

            if isinstance(value, bool):
                continue

            values.append(str(value))

        return " ".join(values)

    return str(item)

In [56]:
def clean_list_items(items):
    cleaned_items = []

    for item in ensure_list(items):
        item_text = extract_text_from_item(item).strip()

        if item_text == "":
            continue

        if item_text.lower() in ["none", "null", "nan", "unknown", "n/a"]:
            continue

        cleaned_items.append(item_text)

    return cleaned_items

In [57]:
def unique_list(items):
    seen = set()
    unique_items = []

    for item in items:
        normalized_item = normalize_text(item)

        if normalized_item == "":
            continue

        if normalized_item not in seen:
            seen.add(normalized_item)
            unique_items.append(item)

    return unique_items

## 6. Build Searchable CV Evidence

In [58]:
cv_skill_fields = [
    "technical_skills",
    "programming_languages",
    "frameworks_and_libraries",
    "databases",
    "cloud_and_devops_tools",
    "data_and_ai_tools",
    "testing_tools",
    "other_tools",
    "soft_skills"
]

cv_text_evidence_fields = [
    "profile_summary",
    "technical_skills",
    "programming_languages",
    "frameworks_and_libraries",
    "databases",
    "cloud_and_devops_tools",
    "data_and_ai_tools",
    "testing_tools",
    "other_tools",
    "soft_skills",
    "education",
    "work_experience",
    "projects",
    "certifications",
    "languages"
]

In [59]:
cv_all_skills = []

for field in cv_skill_fields:
    cv_all_skills.extend(clean_list_items(structured_cv.get(field, [])))

cv_all_skills = unique_list(cv_all_skills)

In [60]:
cv_evidence_parts = []

for field in cv_text_evidence_fields:
    values = clean_list_items(structured_cv.get(field, []))
    cv_evidence_parts.extend(values)

In [61]:
searchable_cv_text = normalize_text(" ".join(cv_evidence_parts))

In [62]:
print(f"Number of extracted CV skills: {len(cv_all_skills)}")
print(f"Length of searchable CV evidence text: {len(searchable_cv_text)}")

Number of extracted CV skills: 13
Length of searchable CV evidence text: 1892


## 7. Extract Job Requirements

In [63]:
job_required_skills = clean_list_items(structured_job.get("required_skills", []))
job_nice_to_have_skills = clean_list_items(structured_job.get("nice_to_have_skills", []))

In [64]:
job_technology_fields = [
    "programming_languages",
    "frameworks_and_libraries",
    "databases",
    "cloud_and_devops_tools",
    "data_and_ai_tools",
    "testing_tools",
    "other_tools"
]

In [65]:
job_technology_skills = []

for field in job_technology_fields:
    job_technology_skills.extend(clean_list_items(structured_job.get(field, [])))

In [66]:
job_required_skills = unique_list(job_required_skills)
job_nice_to_have_skills = unique_list(job_nice_to_have_skills)
job_technology_skills = unique_list(job_technology_skills)

In [67]:
print(f"Required skills: {len(job_required_skills)}")
print(f"Nice-to-have skills: {len(job_nice_to_have_skills)}")
print(f"Technology skills: {len(job_technology_skills)}")

Required skills: 15
Nice-to-have skills: 1
Technology skills: 10


## 8. Define Rule-Based Matching Function

In [68]:
def match_items(cv_items, job_items, searchable_cv_text):
    cv_normalized_map = {}

    for cv_item in cv_items:
        normalized_cv_item = normalize_text(cv_item)

        if normalized_cv_item:
            cv_normalized_map[normalized_cv_item] = cv_item

    matched_items = []
    missing_items = []

    for job_item in job_items:
        normalized_job_item = normalize_text(job_item)

        if normalized_job_item == "":
            continue

        if normalized_job_item in cv_normalized_map:
            matched_items.append({
                "job_requirement": job_item,
                "cv_evidence": cv_normalized_map[normalized_job_item],
                "match_type": "exact_skill_match"
            })

        elif contains_normalized_phrase(searchable_cv_text, normalized_job_item):
            matched_items.append({
                "job_requirement": job_item,
                "cv_evidence": "Found in structured CV evidence text",
                "match_type": "text_evidence_match"
            })

        else:
            missing_items.append(job_item)

    return matched_items, missing_items

## 9. Match Required, Nice-to-Have and Technology Skills

In [69]:
matched_required_skills, missing_required_skills = match_items(
    cv_all_skills,
    job_required_skills,
    searchable_cv_text
)

matched_nice_to_have_skills, missing_nice_to_have_skills = match_items(
    cv_all_skills,
    job_nice_to_have_skills,
    searchable_cv_text
)

matched_technology_skills, missing_technology_skills = match_items(
    cv_all_skills,
    job_technology_skills,
    searchable_cv_text
)

In [70]:
print(f"Matched required skills: {len(matched_required_skills)}")
print(f"Missing required skills: {len(missing_required_skills)}")
print(f"Matched nice-to-have skills: {len(matched_nice_to_have_skills)}")
print(f"Missing nice-to-have skills: {len(missing_nice_to_have_skills)}")
print(f"Matched technology skills: {len(matched_technology_skills)}")
print(f"Missing technology skills: {len(missing_technology_skills)}")

Matched required skills: 0
Missing required skills: 15
Matched nice-to-have skills: 0
Missing nice-to-have skills: 1
Matched technology skills: 0
Missing technology skills: 10


## 10. Calculate Skill-Based Scores

In [71]:
def calculate_match_percentage(matched_items, total_items):
    if total_items == 0:
        return 100

    return round((len(matched_items) / total_items) * 100, 2)


In [72]:
required_skills_score = calculate_match_percentage(
    matched_required_skills,
    len(job_required_skills)
)

nice_to_have_score = calculate_match_percentage(
    matched_nice_to_have_skills,
    len(job_nice_to_have_skills)
)

technology_score = calculate_match_percentage(
    matched_technology_skills,
    len(job_technology_skills)
)

In [73]:
required_skills_score, nice_to_have_score, technology_score

(0.0, 0.0, 0.0)

## 11. Extract Experience Requirements

In [74]:
def extract_number_from_text(text):
    if text is None:
        return None

    match = re.search(r"\d+", str(text))

    if match:
        return int(match.group())

    return None

In [75]:
def get_job_minimum_years(structured_job):
    experience_requirements = structured_job.get("experience_requirements", [])

    if not isinstance(experience_requirements, list):
        return None

    years = []

    for requirement in experience_requirements:
        if not isinstance(requirement, dict):
            continue

        minimum_years = extract_number_from_text(requirement.get("minimum_years"))

        if minimum_years is not None:
            years.append(minimum_years)

    if len(years) == 0:
        return None

    return max(years)

In [76]:
def get_cv_years_of_experience(structured_cv):
    possible_fields = [
        "years_of_experience",
        "total_years_of_experience",
        "professional_experience_years"
    ]

    for field in possible_fields:
        years = extract_number_from_text(structured_cv.get(field))

        if years is not None:
            return years

    return None

## 12. Calculate Experience Score

In [77]:
job_minimum_years = get_job_minimum_years(structured_job)
cv_years_of_experience = get_cv_years_of_experience(structured_cv)

In [78]:
if job_minimum_years is None:
    experience_score = 100
    experience_note = "The job posting does not clearly specify minimum years of experience."

elif cv_years_of_experience is None:
    experience_score = 0
    experience_note = "The job posting specifies experience requirements, but years of experience are not clearly evidenced in the CV."

elif cv_years_of_experience >= job_minimum_years:
    experience_score = 100
    experience_note = "The CV meets or exceeds the minimum years of experience requirement."

else:
    experience_score = round((cv_years_of_experience / job_minimum_years) * 100, 2)
    experience_note = "The CV shows fewer years of experience than required by the job posting."

## 13. Calculate Education Score

In [79]:
job_education_requirements = clean_list_items(structured_job.get("education_requirements", []))
cv_education = clean_list_items(structured_cv.get("education", []))

In [80]:
if len(job_education_requirements) == 0:
    education_score = 100
    education_note = "The job posting does not clearly specify education requirements."

elif len(cv_education) == 0:
    education_score = 0
    education_note = "The job posting specifies education requirements, but education is not clearly evidenced in the CV."

else:
    education_score = 70
    education_note = "Education is present in the CV, but detailed equivalence should be reviewed manually."

## 14. Match Certifications and Language Requirements

In [81]:
cv_certifications = clean_list_items(structured_cv.get("certifications", []))
job_certifications = clean_list_items(structured_job.get("certifications", []))

cv_languages = clean_list_items(structured_cv.get("languages", []))
job_language_requirements = clean_list_items(structured_job.get("language_requirements", []))

In [82]:
matched_certifications, missing_certifications = match_items(
    cv_certifications,
    job_certifications,
    searchable_cv_text
)

matched_languages, missing_languages = match_items(
    cv_languages,
    job_language_requirements,
    searchable_cv_text
)

certification_score = calculate_match_percentage(
    matched_certifications,
    len(job_certifications)
)

language_score = calculate_match_percentage(
    matched_languages,
    len(job_language_requirements)
)

## 15. Calculate Rule-Based Score

In [83]:
rule_based_score_weights = {
    "required_skills_score": 0.40,
    "technology_score": 0.25,
    "experience_score": 0.20,
    "education_score": 0.08,
    "nice_to_have_score": 0.04,
    "certification_score": 0.02,
    "language_score": 0.01
}

In [84]:
rule_based_score = (
    required_skills_score * rule_based_score_weights["required_skills_score"] +
    technology_score * rule_based_score_weights["technology_score"] +
    experience_score * rule_based_score_weights["experience_score"] +
    education_score * rule_based_score_weights["education_score"] +
    nice_to_have_score * rule_based_score_weights["nice_to_have_score"] +
    certification_score * rule_based_score_weights["certification_score"] +
    language_score * rule_based_score_weights["language_score"]
)

In [85]:
rule_based_score = round(rule_based_score, 2)

## 16. LLM-Based Semantic Matching

In [86]:
class SemanticMatchingAnalysis(BaseModel):

    responsibilities_alignment_score: int = Field(
        ge=0,
        le=100,
        description=(
            "Score from 0 to 100 for semantic alignment between the candidate's CV and the job responsibilities. "
            "Evaluate whether the candidate's work experience, projects, activities and described responsibilities "
            "show evidence of being able to perform the responsibilities listed in the job posting. "
            "Use only evidence explicitly present in the CV. Do not invent experience or assume ability without evidence. "
            "0-20: CV does not show relevant evidence for the listed job responsibilities. "
            "21-40: CV shows very limited or weak indirect evidence for only a few responsibilities. "
            "41-60: CV shows partial alignment with some responsibilities, but important responsibility areas are missing or unclear. "
            "61-80: CV shows clear alignment with many responsibilities, with some gaps or only indirect evidence. "
            "81-100: CV strongly aligns with most responsibilities through clear experience, projects or described activities."
        )
    )

    soft_skills_evidence_score: int = Field(
        ge=0,
        le=100,
        description=(
            "Score from 0 to 100 for evidence of soft skills requested in the job posting. "
            "Soft skills do not need to appear as exact phrases in the CV. Evaluate whether they are supported by "
            "concrete CV evidence such as teamwork, mentoring, leadership, client or stakeholder communication, "
            "documentation, collaboration, problem solving, ownership of tasks or project coordination. "
            "Use only evidence from the CV. Do not assume that the candidate has a soft skill if it is not evidenced. "
            "0-20: CV does not provide clear evidence for the requested soft skills. "
            "21-40: CV provides weak or very limited indirect evidence for a few soft skills. "
            "41-60: CV provides partial evidence for some soft skills, but several remain unclear. "
            "61-80: CV provides good evidence for many requested soft skills through experience or projects. "
            "81-100: CV strongly evidences most requested soft skills through concrete experience, responsibilities or achievements."
        )
    )

    role_fit_summary: str = Field(
        description=(
            "Concise evidence-based summary of the overall semantic fit between the CV and the job posting. "
            "Mention the strongest areas of alignment and the most important gaps. "
            "Do not mention skills, experience, projects or achievements that are not evidenced in the CV."
        )
    )

    responsibilities_evidenced: List[str] = Field(
        default_factory=list,
        description=(
            "List of job responsibilities that are clearly or reasonably indirectly evidenced in the CV. "
            "Each item should reference a responsibility from the job posting and briefly explain what CV evidence supports it. "
            "Only include responsibilities supported by CV content."
        )
    )

    responsibilities_not_evidenced: List[str] = Field(
        default_factory=list,
        description=(
            "List of job responsibilities from the job posting that are not clearly evidenced in the CV. "
            "Use this field for responsibilities, duties or role expectations that cannot be confirmed from the CV. "
            "Do not include responsibilities already listed as evidenced."
        )
    )

    soft_skills_evidenced: List[str] = Field(
        default_factory=list,
        description=(
            "List of soft skills from the job posting that are clearly or indirectly evidenced in the CV. "
            "A soft skill may be considered evidenced if the CV shows concrete support such as teamwork, mentoring, "
            "leadership, communication with clients or stakeholders, documentation, collaboration, problem solving "
            "or ownership of tasks. Do not include generic soft skills without CV-based evidence."
        )
    )

    soft_skills_not_clearly_evidenced: List[str] = Field(
        default_factory=list,
        description=(
            "List of soft skills from the job posting that are not clearly evidenced in the CV. "
            "These should not be treated as strict missing technical skills, but as soft skills without enough CV evidence. "
            "Do not include a soft skill here if it was already listed in soft_skills_evidenced."
        )
    )

    evidence_notes: List[str] = Field(
        default_factory=list,
        description=(
            "Important notes explaining how the semantic matching decision was made. "
            "Notes should explain direct evidence, indirect evidence, weak evidence, uncertainty or important limitations "
            "of the comparison. Use only CV and job posting evidence. Do not invent candidate abilities or achievements."
        )
    )

## 17. Define LLM Semantic Matching Prompt

In [87]:
semantic_matching_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI assistant specialized in semantic CV-job matching.

Your task is to compare a structured CV with a structured IT job posting.

Important rules:
- Use only the information provided in the structured CV and structured job posting.
- Do not invent skills, experience, projects, certifications, education or achievements.
- Do not assume that the candidate has a skill if it is not evidenced in the CV.
- Do not override the rule-based technical matching result.
- Focus only on semantic alignment of responsibilities, soft skills and overall role fit.
- Soft skills do not need to appear as exact phrases in the CV.
- A soft skill may be considered evidenced if the CV shows relevant experience, projects, teamwork, leadership, communication with stakeholders, mentoring, documentation, problem solving or ownership of tasks.
- If a soft skill is not clearly evidenced, mark it as not clearly evidenced instead of calling it strictly missing.
- Be conservative and evidence-based.
- Return the result using the required structured schema.
"""
        ),
        (
            "human",
            """
Structured CV:

{structured_cv}

Structured job posting:

{structured_job}

Rule-based missing required skills:

{missing_required_skills}

Rule-based missing technology skills:

{missing_technology_skills}

Perform semantic matching analysis.
"""
        )
    ]
)

## 18. Run LLM Semantic Matching

In [88]:
structured_semantic_llm = llm.with_structured_output(SemanticMatchingAnalysis)

In [89]:
semantic_matching_chain = semantic_matching_prompt | structured_semantic_llm

In [90]:
semantic_matching_result = semantic_matching_chain.invoke(
    {
        "structured_cv": json.dumps(structured_cv, indent=2, ensure_ascii=False),
        "structured_job": json.dumps(structured_job, indent=2, ensure_ascii=False),
        "missing_required_skills": json.dumps(missing_required_skills, indent=2, ensure_ascii=False),
        "missing_technology_skills": json.dumps(missing_technology_skills, indent=2, ensure_ascii=False)
    }
)

In [91]:
semantic_matching_dict = semantic_matching_result.model_dump()

## 19. Calculate LLM Semantic Score

In [92]:
semantic_score = (
    semantic_matching_dict["responsibilities_alignment_score"] * 0.70 +
    semantic_matching_dict["soft_skills_evidence_score"] * 0.30
)

In [93]:
semantic_score = round(semantic_score, 2)

## 20.Calculate Final Hybrid Match Score

In [94]:
final_hybrid_score = (
    rule_based_score * 0.85 +
    semantic_score * 0.15
)

In [95]:
final_hybrid_score = round(final_hybrid_score, 2)

## 21. Define Match Category

In [96]:
def define_match_category(score):
    if score >= 85:
        return "Strong match"
    elif score >= 70:
        return "Good match"
    elif score >= 50:
        return "Partial match"
    else:
        return "Weak match"

In [97]:
match_category = define_match_category(final_hybrid_score)

## 22. Create Matching Result Dictionary

In [98]:
matching_result = {
    "metadata": {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "matching_type": "hybrid_rule_based_and_llm_semantic_matching",
        "cv_source": str(structured_cv_path),
        "job_source": str(structured_job_path),
        "llm_model": "gpt-4o-mini"
    },
    "job_information": {
        "job_title": structured_job.get("job_title"),
        "company_name": structured_job.get("company_name"),
        "job_category": structured_job.get("job_category"),
        "location": structured_job.get("location"),
        "work_mode": structured_job.get("work_mode"),
        "employment_type": structured_job.get("employment_type")
    },
    "final_result": {
        "final_hybrid_score": final_hybrid_score,
        "rule_based_score": rule_based_score,
        "semantic_score": semantic_score,
        "match_category": match_category
    },
    "score_breakdown": {
        "required_skills_score": required_skills_score,
        "technology_score": technology_score,
        "experience_score": experience_score,
        "education_score": education_score,
        "nice_to_have_score": nice_to_have_score,
        "certification_score": certification_score,
        "language_score": language_score,
        "rule_based_score": rule_based_score,
        "semantic_score": semantic_score,
        "final_hybrid_score": final_hybrid_score
    },
    "matched_items": {
        "matched_required_skills": matched_required_skills,
        "matched_nice_to_have_skills": matched_nice_to_have_skills,
        "matched_technology_skills": matched_technology_skills,
        "matched_certifications": matched_certifications,
        "matched_languages": matched_languages
    },
    "missing_items": {
        "missing_required_skills": missing_required_skills,
        "missing_nice_to_have_skills": missing_nice_to_have_skills,
        "missing_technology_skills": missing_technology_skills,
        "missing_certifications": missing_certifications,
        "missing_languages": missing_languages
    },
    "experience_analysis": {
        "job_minimum_years": job_minimum_years,
        "cv_years_of_experience": cv_years_of_experience,
        "experience_score": experience_score,
        "experience_note": experience_note
    },
    "education_analysis": {
        "job_education_requirements": job_education_requirements,
        "cv_education": cv_education,
        "education_score": education_score,
        "education_note": education_note
    },
    "semantic_analysis": semantic_matching_dict
}

In [99]:
matching_result

{'metadata': {'created_at': '2026-07-19T00:21:14',
  'matching_type': 'hybrid_rule_based_and_llm_semantic_matching',
  'cv_source': '..\\outputs\\cv_extraction\\structured_cv.json',
  'job_source': '..\\outputs\\job_extraction\\structured_job.json',
  'llm_model': 'gpt-4o-mini'},
 'job_information': {'job_title': 'Senior Developer',
  'company_name': 'USLI',
  'job_category': 'Software Development',
  'location': None,
  'work_mode': 'On-site 75% of the time',
  'employment_type': 'Full-time'},
 'final_result': {'final_hybrid_score': 12.15,
  'rule_based_score': 8.6,
  'semantic_score': 32.3,
  'match_category': 'Weak match'},
 'score_breakdown': {'required_skills_score': 0.0,
  'technology_score': 0.0,
  'experience_score': 0,
  'education_score': 70,
  'nice_to_have_score': 0.0,
  'certification_score': 100,
  'language_score': 100,
  'rule_based_score': 8.6,
  'semantic_score': 32.3,
  'final_hybrid_score': 12.15},
 'matched_items': {'matched_required_skills': [],
  'matched_nice_to

## 23. Save Matching Result as JSON

In [100]:
with open(matching_result_path, "w", encoding="utf-8") as file:
    json.dump(matching_result, file, indent=4, ensure_ascii=False)

## 24. Generate Markdown Matching Report

In [101]:
def create_markdown_list(items, key=None, empty_message="- No items found."):
    if not items:
        return empty_message

    lines = []

    for item in items:
        if isinstance(item, dict) and key is not None:
            value = item.get(key)
        else:
            value = item

        if value is not None and str(value).strip() != "":
            lines.append(f"- {value}")

    if not lines:
        return empty_message

    return "\n".join(lines)

In [102]:
report = f"""
# CV-Job Matching Report

## Job Information

- Job title: {structured_job.get("job_title")}
- Company: {structured_job.get("company_name")}
- Job category: {structured_job.get("job_category")}
- Location: {structured_job.get("location")}
- Work mode: {structured_job.get("work_mode")}
- Employment type: {structured_job.get("employment_type")}

## Final Hybrid Matching Result

- Final hybrid match score: {final_hybrid_score}/100
- Match category: {match_category}
- Rule-based score: {rule_based_score}/100
- LLM semantic score: {semantic_score}/100

## Rule-Based Score Breakdown

- Required skills score: {required_skills_score}/100
- Technology score: {technology_score}/100
- Experience score: {experience_score}/100
- Education score: {education_score}/100
- Nice-to-have score: {nice_to_have_score}/100
- Certification score: {certification_score}/100
- Language score: {language_score}/100

## Matched Required Skills

{create_markdown_list(matched_required_skills, key="job_requirement", empty_message="- No required skills clearly matched.")}

## Missing Required Skills

{create_markdown_list(missing_required_skills, empty_message="- No required skills are missing.")}

## Matched Nice-to-Have Skills

{create_markdown_list(matched_nice_to_have_skills, key="job_requirement", empty_message="- No nice-to-have skills clearly matched.")}

## Missing Nice-to-Have Skills

{create_markdown_list(missing_nice_to_have_skills, empty_message="- No nice-to-have skills are missing.")}

## Matched Technology Skills

{create_markdown_list(matched_technology_skills, key="job_requirement", empty_message="- No technology skills clearly matched.")}

## Missing Technology Skills

{create_markdown_list(missing_technology_skills, empty_message="- No technology skills are missing.")}

## Experience Analysis

- Job minimum years: {job_minimum_years}
- CV years of experience: {cv_years_of_experience}
- Experience score: {experience_score}/100
- Note: {experience_note}

## Education Analysis

- Education score: {education_score}/100
- Note: {education_note}

## Certifications

### Matched Certifications

{create_markdown_list(matched_certifications, key="job_requirement", empty_message="- No certifications clearly matched.")}

### Missing Certifications

{create_markdown_list(missing_certifications, empty_message="- No certifications are missing.")}

## Language Requirements

### Matched Languages

{create_markdown_list(matched_languages, key="job_requirement", empty_message="- No language requirements clearly matched.")}

### Missing Languages

{create_markdown_list(missing_languages, empty_message="- No language requirements are missing.")}

## LLM Semantic Analysis

### Role Fit Summary

{semantic_matching_dict["role_fit_summary"]}

### Responsibilities Evidenced in CV

{create_markdown_list(semantic_matching_dict["responsibilities_evidenced"], empty_message="- No responsibilities clearly evidenced.")}

### Responsibilities Not Clearly Evidenced

{create_markdown_list(semantic_matching_dict["responsibilities_not_evidenced"], empty_message="- No responsibility gaps identified.")}

### Soft Skills Evidenced in CV

{create_markdown_list(semantic_matching_dict["soft_skills_evidenced"], empty_message="- No soft skills clearly evidenced.")}

### Soft Skills Not Clearly Evidenced in CV

{create_markdown_list(semantic_matching_dict["soft_skills_not_clearly_evidenced"], empty_message="- No soft skill gaps identified.")}

### Evidence Notes

{create_markdown_list(semantic_matching_dict["evidence_notes"], empty_message="- No additional evidence notes.")}
"""

## 25. Save Markdown Matching Report

In [103]:
with open(matching_report_path, "w", encoding="utf-8") as file:
    file.write(report)

In [104]:
print(report)


# CV-Job Matching Report

## Job Information

- Job title: Senior Developer
- Company: USLI
- Job category: Software Development
- Location: None
- Work mode: On-site 75% of the time
- Employment type: Full-time

## Final Hybrid Matching Result

- Final hybrid match score: 12.15/100
- Match category: Weak match
- Rule-based score: 8.6/100
- LLM semantic score: 32.3/100

## Rule-Based Score Breakdown

- Required skills score: 0.0/100
- Technology score: 0.0/100
- Experience score: 0/100
- Education score: 70/100
- Nice-to-have score: 0.0/100
- Certification score: 100/100
- Language score: 100/100

## Matched Required Skills

- No required skills clearly matched.

## Missing Required Skills

- ASP.NET
- C#
- MVC
- Object Oriented Design/Development
- VB.NET
- Agile
- SCRUM
- JavaScript
- jQuery
- JSON
- Knockout.js
- Angular.js
- HTML5
- SQL Server
- SQL/XML

## Matched Nice-to-Have Skills

- No nice-to-have skills clearly matched.

## Missing Nice-to-Have Skills

- Emerging Microsoft 